In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import importlib

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))

import src.option_filter_polygon as ofp
importlib.reload(ofp)
from src.option_filter_polygon import filter_option_chain

import src.IV_Finder as ivf
importlib.reload(ivf)
from src.IV_Finder import implied_volatility

RAW_DATA_DIR       = project_root / "data" / "raw"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
FIGURES_DIR        = project_root / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Grid parameters
LOG_MONEYNESS_GRID = np.linspace(np.log(0.80), np.log(1.20), 50)
TTM_GRID           = np.linspace(14/365, 1.0, 50)

print("Config loaded.")
print(f"Raw data dir:       {RAW_DATA_DIR}")
print(f"Processed data dir: {PROCESSED_DATA_DIR}")
print(f"Figures dir:        {FIGURES_DIR}")

Config loaded.
Raw data dir:       /Users/kennethdavis/Desktop/Personal/Personal Projects/IV Surface/Implied-Volatility-Surface-PCA/data/raw
Processed data dir: /Users/kennethdavis/Desktop/Personal/Personal Projects/IV Surface/Implied-Volatility-Surface-PCA/data/processed
Figures dir:        /Users/kennethdavis/Desktop/Personal/Personal Projects/IV Surface/Implied-Volatility-Surface-PCA/figures


In [2]:
import yfinance as yf

def get_risk_free_rate(date: str) -> float:
    """
    Fetch the 13-week T-bill rate (^IRX) for a given date from yfinance.
    ^IRX is quoted as a percentage so we divide by 100.
    Falls back to the most recent available rate if exact date not found.
    """
    start = (pd.Timestamp(date) - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    end   = (pd.Timestamp(date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    irx = yf.download("^IRX", start=start, end=end, progress=False)
    if irx.empty:
        raise ValueError(f"No ^IRX data found near {date}")
    return float(irx["Close"].iloc[-1]) / 100

In [3]:
def nadaraya_watson(m_grid, t_grid, m_data, t_data, iv_data, h_m=0.05, h_t=0.1):
    """
    Nadaraya-Watson kernel smoother over (log-moneyness, TTM) space.
    Following Cont & da Fonseca (2002).
    """
    surface = np.zeros((len(t_grid), len(m_grid)))
    for i, t in enumerate(t_grid):
        for j, m in enumerate(m_grid):
            w = (np.exp(-0.5 * ((m_data - m) / h_m)**2) *
                 np.exp(-0.5 * ((t_data - t) / h_t)**2))
            if w.sum() > 0:
                surface[i, j] = np.dot(w, iv_data) / w.sum()
            else:
                surface[i, j] = np.nan
    return surface

In [4]:
def calculate_ivs(df: pd.DataFrame, spot: float, r: float) -> pd.DataFrame:
    """
    Calculate implied volatility for each contract in the filtered DataFrame.
    Returns DataFrame with IV column added, NaN rows dropped.
    """
    ivs = []
    for _, row in df.iterrows():
        iv = implied_volatility(
            row["mid"],
            spot,
            row["strike"],
            row["TTM"],
            r,
            row["option_type"]
        )
        ivs.append(iv)
    df = df.copy()
    df["IV"] = ivs
    df = df.dropna(subset=["IV"])
    df = df[df["IV"] > 0]
    return df

In [5]:
def save_surface_figure(IV_surface, date: str):
    """
    Save a 3D IV surface plot to the figures directory.
    """
    M, T = np.meshgrid(LOG_MONEYNESS_GRID, TTM_GRID)
    fig = plt.figure(figsize=(12, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(M, T, IV_surface, cmap='viridis', alpha=0.8)
    ax.set_xlabel("Log-Moneyness (log(K/S))")
    ax.set_ylabel("Time to Maturity (years)")
    ax.set_zlabel("Implied Volatility")
    ax.set_title(f"SPY IV Surface — {date}")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"IV_surface_{date}.png", dpi=150)
    plt.close(fig)

In [6]:
skipped = []
completed = []

raw_files = sorted(RAW_DATA_DIR.glob("SPY_*.csv"))
print(f"Found {len(raw_files)} raw files to process\n")

for raw_file in raw_files:
    date = raw_file.stem.replace("SPY_", "")
    out_path = PROCESSED_DATA_DIR / f"SPY_IV_{date}.csv"

    # Skip if already processed
    if out_path.exists():
        print(f"[SKIP] {date} already processed")
        completed.append(date)
        continue

    try:
        # Load raw data
        df_raw = pd.read_csv(raw_file)
        spot = df_raw["spot"].iloc[0]

        # Get risk-free rate
        r = get_risk_free_rate(date)

        # Filter
        df_filtered = filter_option_chain(df_raw, S=spot, target_date=date)
        if len(df_filtered) < 50:
            print(f"[WARN] {date} — only {len(df_filtered)} contracts after filtering, skipping")
            skipped.append(date)
            continue

        # Calculate IV
        df_iv = calculate_ivs(df_filtered, spot=spot, r=r)
        if len(df_iv) < 50:
            print(f"[WARN] {date} — only {len(df_iv)} contracts with valid IV, skipping")
            skipped.append(date)
            continue

        # Build surface
        IV_surface = nadaraya_watson(
            LOG_MONEYNESS_GRID, TTM_GRID,
            df_iv["log_moneyness"].values,
            df_iv["TTM"].values,
            df_iv["IV"].values
        )

        # Save processed DataFrame
        df_iv.to_csv(out_path, index=False)

        # Save figure
        save_surface_figure(IV_surface, date)

        completed.append(date)
        print(f"[OK]   {date} — {len(df_iv)} contracts, surface saved")

    except Exception as e:
        print(f"[ERR]  {date} — {e}")
        skipped.append(date)

print(f"\nDone. {len(completed)} days completed, {len(skipped)} skipped/failed")
if skipped:
    print(f"Skipped: {skipped}")

Found 248 raw files to process



/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_49468/3886535064.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(irx["Close"].iloc[-1]) / 100
/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_49468/3886535064.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(irx["Close"].iloc[-1]) / 100


[WARN] 2025-01-02 — only 0 contracts after filtering, skipping
[WARN] 2025-01-03 — only 0 contracts after filtering, skipping


/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_49468/3886535064.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(irx["Close"].iloc[-1]) / 100


[WARN] 2025-01-06 — only 0 contracts after filtering, skipping


...

[OK]   2025-12-26 — 3684 contracts, surface saved


/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_49468/3886535064.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(irx["Close"].iloc[-1]) / 100


[OK]   2025-12-29 — 3802 contracts, surface saved


/var/folders/st/w6lpxhcj7lq3198_dz6d5w_00000gn/T/ipykernel_49468/3886535064.py:14: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(irx["Close"].iloc[-1]) / 100


[OK]   2025-12-30 — 3854 contracts, surface saved

Done. 199 days completed, 49 skipped/failed
Skipped: ['2025-01-02', '2025-01-03', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-10', '2025-01-13', '2025-01-14', '2025-01-15', '2025-01-16', '2025-01-17', '2025-01-21', '2025-01-22', '2025-01-23', '2025-01-24', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-03', '2025-02-04', '2025-02-05', '2025-02-06', '2025-02-07', '2025-02-10', '2025-02-11', '2025-02-12', '2025-02-13', '2025-02-14', '2025-02-18', '2025-02-19', '2025-02-20', '2025-02-21', '2025-02-24', '2025-02-25', '2025-02-26', '2025-02-27', '2025-02-28', '2025-03-03', '2025-03-04', '2025-03-05', '2025-03-06', '2025-03-07', '2025-03-10', '2025-03-11', '2025-03-12', '2025-03-13', '2025-03-14']


In [7]:
processed_files = sorted(PROCESSED_DATA_DIR.glob("SPY_IV_*.csv"))
figures = sorted(FIGURES_DIR.glob("IV_surface_*.png"))

print(f"Processed CSVs: {len(processed_files)}")
print(f"Figures saved:  {len(figures)}")

# Spot check one file
sample = pd.read_csv(processed_files[0])
print(f"\nSample columns: {list(sample.columns)}")
print(f"Sample rows: {len(sample)}")
print(sample[["strike", "option_type", "TTM", "log_moneyness", "mid", "IV"]].head())

Processed CSVs: 199
Figures saved:  199

Sample columns: ['openInterest', 'day.change', 'day.change_percent', 'day.close', 'day.high', 'day.last_updated', 'day.low', 'day.open', 'day.previous_close', 'volume', 'mid', 'option_type', 'details.exercise_style', 'expiration_date', 'details.shares_per_contract', 'strike', 'details.ticker', 'underlying_asset.ticker', 'implied_volatility', 'greeks.delta', 'greeks.gamma', 'greeks.theta', 'greeks.vega', 'spot', 'trade_date', 'moneyness', 'log_moneyness', 'TTM', 'atm_dist', 'IV']
Sample rows: 138
   strike option_type      TTM  log_moneyness        mid        IV
0   565.0         put  1.00000      -0.003798    0.05429  0.020662
1   565.0        call  1.00000      -0.003798   98.47500  0.387706
2   570.0         put  1.00000       0.005013    0.05583  0.017242
3   570.0        call  0.99726       0.005013  108.07000  0.443056
4   560.0        call  0.99726      -0.012687  118.04000  0.469503
